# Regular expressions

Regular expressions are most useful when there is a pattern to data

Typos and OCR errors can be worked around  
But difficult to use regular expressions without some pattern  

Depending on the intention, the pattern can be quite loose. 

We will look at a couple of examples of regex coding I have done to get a better sense of how it works

At least in social science a difficulty with teaching regular expressions is that the closer regex code is to development stage, the less impressive it looks.

Regular expressions are built up by finding patterns that match and then adding in the close variants.  


# Central Bank Coding example

In this example, we will use regular expressions to parse some files  
For the last twenty years, I have been keeping track of the CBI codings I have done in Word documents.  
I have converted these to txt files and want to split the codings into a dataframe  

```
Objectives (weight = .15)
Angola 1997: The main objective of the Bank shall be to ensure the preservation of the value of the national currency. (Article 3) (Coded as .8)
Angola 2010: 1. As the central and issuing bank, the National Bank of Angola, ensures the preservation of the value of the national currency and participates in the definition of the monetary, financial and foreign exchange policies.
2. Notwithstanding the preceding paragraph, execution, monitoring and control of monetary, exchange and credit, the management of the payment system and management of the currency under the economic policy of the country are incumbent upon the National Bank of Angola. (Article 3) (coded as .6)
```

**Regular expresions** are great at finding _patterns_ in text. I set up my files in a very similar manner:
- **country name** all in caps at the beginning
- **indicator name** name of each indicator
- **country-year** the year of the country's law
- **article text**
- **article number**
- **coding** the score I gave the article

In the code below we will use this pattern to extract the documents into separate fields


# Other examples: Federal gifts

US federal employees are required to notify their employing agency of any gifts they've received from foreign sources.  
The gifts should not be kept but should be disposed of in some way  

We can use regular expressions parse the reported gifts:  
<PRTPAGE P="3952"/>
<ENT I="01">The Honorable Joseph R. Biden Jr., President of the United States</ENT>
<ENT>Notre Dame Fragment Sculpture, WWII Warship Ensign, “France Forever” Posters, Josephine Baker Painting by Raphaël Barontini, “MÉMOIRES DE GUERRE” by Charles de Gaulle Book Set, Normandy Landings Photograph by Robert Capa, Statue of Liberty Glass Replica Sculpture, “The Complete War Memoirs of Charles De Gaulle” Book. Rec'd—6/8/2024. Est. Value—$3,070.00. Disposition—Waiting to receive from USSS (Sculpture); Transferred to NARA (Wood Sculpture, Ensign, Photograph, Glass Sculpture, Posters, Painting, Book Set, and Book)</ENT>
<ENT>His Excellency Emmanuel Macron, President of the French Republic</ENT>
<ENT>Non-acceptance would cause embarrassment to donor and U.S.</ENT>

The `<ENT>` tags will allow us to split the gift into 
- **recipient** (`<ENT I="01">The Honorable Joseph R. Biden Jr., President of the United States</ENT>`)
- **gift** (`<ENT>Notre Dame Fragment Sculpture, WWII Warship Ensign, “France Forever” Posters,`)
- **giver** (`<ENT>His Excellency Emmanuel Macron, President of the French Republic</ENT>`)
- **reason** (`<ENT>Non-acceptance would cause embarrassment to donor and U.S.</ENT>`)

Then we want to break out these parts of the gift: `Rec'd—6/8/2024. Est. Value—$3,070.00. Disposition—`  



In [1]:
import re
import pandas as pd
import os

In [2]:
eslist = []
allcodings = []
files = os.listdir('c:/Users/arpie/Dropbox/CBtexts/codings/')
txt_files = [f for f in files if f.endswith('.txt')]
txt_files

['bookscores.txt',
 'bookscores_pt2.txt',
 'bookscores_pt3.txt',
 'scores2021_africa.txt',
 'scores2021_asia.txt',
 'scores2021_easteur.txt',
 'scores2021_mideast.txt',
 'scores2021_pt1.txt',
 'scoreslatam2021.txt']

In [3]:
#f = txt_files[0]
f = 'bookscores.txt'
#with open('c:/Users/arpie/Dropbox/CBtexts/codings/'+f,encoding='utf-8') as nf:
#    t = nf.read()
with open(f,encoding='utf-8') as nf:
    t = nf.read()


In [4]:
print(t[0:1000])

CODING SCHEMES FOR CBI INDEXES, pt. 2
Cukierman, Webb, and Neyapti (1994)

ANGOLA
Chief Executive Officer (weight = .20)
Term of office
Angola 1997: The Governor shall be appointed by a decree of the President of the Republic and perform his duties for a five-years term. (Article 51) (Coded as .5)
Angola 2010: The Governor is appointed by the President of the Republic and shall perform his duties for a period of five years, renewable for equal periods. (article 50) (Coded as .5)
Who appoints CEO?
Angola 1997: The Governor shall be appointed by a decree of the President of the Republic and perform his duties for a five-years term. (Article 51) (Coded as 0)
Angola 2010: The Governor is appointed by the President of the Republic and shall perform his duties for a period of five years, renewable for equal periods. (article 50) (Coded as 0)
Dismissal
(no provision for dismissal in 1997—coded as 1)
Angola 2010: 1. The mandate of the Governor, Deputy Governors and the Directors referred to in

In [5]:
COUNTRY_PATTERN = re.compile(r"(?:^|\n)([A-Z ]{2,})\n")

for match in COUNTRY_PATTERN.finditer(t):
    print("FULL MATCH:", repr(match.group(0)))
    print("COUNTRY:", match.group(1))
    print("SPAN (full):", match.start(), match.end())
    print("SPAN (country only):", match.start(1), match.end(1))
    print("-" * 40)

FULL MATCH: '\nANGOLA\n'
COUNTRY: ANGOLA
SPAN (full): 74 82
SPAN (country only): 75 81
----------------------------------------
FULL MATCH: '\nBAHRAIN\n'
COUNTRY: BAHRAIN
SPAN (full): 12703 12712
SPAN (country only): 12704 12711
----------------------------------------
FULL MATCH: '\nBANGLADESH\n'
COUNTRY: BANGLADESH
SPAN (full): 23374 23386
SPAN (country only): 23375 23385
----------------------------------------
FULL MATCH: '\nBARBADOS\n'
COUNTRY: BARBADOS
SPAN (full): 40467 40477
SPAN (country only): 40468 40476
----------------------------------------
FULL MATCH: '\nBELIZE\n'
COUNTRY: BELIZE
SPAN (full): 51856 51864
SPAN (country only): 51857 51863
----------------------------------------
FULL MATCH: '\nBRUNEI\n'
COUNTRY: BRUNEI
SPAN (full): 63598 63606
SPAN (country only): 63599 63605
----------------------------------------
FULL MATCH: '\nCAMBODIA \n'
COUNTRY: CAMBODIA 
SPAN (full): 69352 69363
SPAN (country only): 69353 69362
----------------------------------------
FULL MATCH: 

# finditer v. findall

I used finditer here rather than findall.  
`finditer` has a couple of advantages in this case.
- Most minor is that it uses an iterator rather than storing all matches
- More important, it returns the start and end positions of the matches as well as the matched group
- findall only returns the matched groups

In [5]:
COUNTRY_PATTERN = re.compile(r"(?:^|\n)([A-Z ]{2,})\n")

def return_countries(text):
    return [
        (m.group(1), m.start(1), m.end(1))
        for m in COUNTRY_PATTERN.finditer(text)
    ]
countries = return_countries(t)
print(countries)

[('ANGOLA', 75, 81), ('BAHRAIN', 12704, 12711), ('BANGLADESH', 23375, 23385), ('BARBADOS', 40468, 40476), ('BELIZE', 51857, 51863), ('BRUNEI', 63599, 63605), ('CAMBODIA ', 69353, 69362), ('CAPE VERDE', 87894, 87904), ('CYPRUS', 96702, 96708), ('EGYPT', 104527, 104532), ('ETHIOPIA', 112996, 113004), ('FIJI', 123430, 123434), ('GAMBIA', 135506, 135512), ('GHANA', 143511, 143516), ('JORDAN', 160687, 160693), ('KUWAIT', 169059, 169065), ('LAOS', 177721, 177725), ('LEBANON', 184732, 184739), ('LIBERIA', 192323, 192330), ('MONTENEGRO', 216217, 216227), ('NAMIBIA', 224361, 224368), ('PAPUA NEW GUINEA', 244749, 244765), ('RWANDA', 259380, 259386), ('SAMOA', 273833, 273838), ('SERBIA', 279250, 279256), ('SEYCHELLES', 302759, 302769), ('SRI LANKA', 312751, 312760), ('SURINAME', 325303, 325311), ('TAIWAN', 335312, 335318), ('TANZANIA', 338016, 338024), ('UGANDA', 374813, 374819), ('ZAMBIA', 398686, 398692)]


In [6]:
COUNTRY_PATTERN = re.compile(r"(?:^|\n)([A-Z ]{2,})\n")

def return_countries(text,pattern):
    countries = []
    for match in pattern.finditer(text):
        country_name = match.group(1)
        start_pos = match.start(1)
        end_pos = match.end(1)

        countries.append((country_name, start_pos, end_pos))

    return countries

countries = return_countries(t, COUNTRY_PATTERN)
print(countries)

[('ANGOLA', 75, 81), ('BAHRAIN', 12704, 12711), ('BANGLADESH', 23375, 23385), ('BARBADOS', 40468, 40476), ('BELIZE', 51857, 51863), ('BRUNEI', 63599, 63605), ('CAMBODIA ', 69353, 69362), ('CAPE VERDE', 87894, 87904), ('CYPRUS', 96702, 96708), ('EGYPT', 104527, 104532), ('ETHIOPIA', 112996, 113004), ('FIJI', 123430, 123434), ('GAMBIA', 135506, 135512), ('GHANA', 143511, 143516), ('JORDAN', 160687, 160693), ('KUWAIT', 169059, 169065), ('LAOS', 177721, 177725), ('LEBANON', 184732, 184739), ('LIBERIA', 192323, 192330), ('MONTENEGRO', 216217, 216227), ('NAMIBIA', 224361, 224368), ('PAPUA NEW GUINEA', 244749, 244765), ('RWANDA', 259380, 259386), ('SAMOA', 273833, 273838), ('SERBIA', 279250, 279256), ('SEYCHELLES', 302759, 302769), ('SRI LANKA', 312751, 312760), ('SURINAME', 325303, 325311), ('TAIWAN', 335312, 335318), ('TANZANIA', 338016, 338024), ('UGANDA', 374813, 374819), ('ZAMBIA', 398686, 398692)]


## Explanation

What do the numbers mean? 

Remember that a string is made up of characters and each character is located at a position in the string.  
The position tells us where in the string that the character appears.  
In this case, the first number is the start of the country name and the last number is the end of the country name  

What this means is that if we extracted the country names correctly, 
the values from the last character of one country to the first value of the next country
are the codings for the former country.  

That is, we know that the codings for Angola start with character 83 and end with character 12702 

We can extract each country's section to then parse the individual codings  

In [7]:
def get_text_slices(cty_spans):
    """ 
    Get slices of text for each country.
    """
    slices = []

    # Iterate over current and next span together
    for (country, start,end), (_, next_start,end) in zip(cty_spans, cty_spans[1:]):
        slices.append((country, t[start:next_start]))

    # Handle last country
    last_country, last_start,end = cty_spans[-1]
    slices.append((last_country, t[last_start:]))

    return slices

In [8]:
cty_slices = get_text_slices(countries)
cty_slices[23]

('SAMOA',
 'SAMOA\nChief Executive Officer (weight = .20)\nTerm of office\nSamoa 2008: (3) The Governor shall be appointed for a period not exceeding three years and on such terms and conditions as may be specified. (Section 9) (Coded as 0) (Changed in 2001)\nWho appoints CEO?\nSamoa 2008: (1) The Head of State, acting on the advice of Cabinet, shall from time to time appoint a Governor of the Central Bank from amongst persons of recognised standing and experience in financial and banking matters. (Section 9) (Coded as .25)\nDismissal\nMay CEO hold other offices in government?\nSamoa 2008: (5) The Governor shall devote the whole of his or her professional services to the Bank and not hold any other office, (whether remunerated or not) without the written permission of the Minister given on the recommendation of the Board. \n(6) No person may be appointed under subsection (1) to be the Governor or may continue to hold office as the Governor, while that person is: \n(a) A Member of Parli

# Indicators

Next we want to do the same withe each of the 16 indicators.  

There are a couple of ways to do this.  
The **staying in control** method is to extract each pattern and then move on to the next  
We begin at the end, see if the pattern is there.  If it is, we extract it and then _cut the pattern from the string_ so the next pattern has less ext to work with. By the end we should have an empty or mostly empty string

A **map and extract** method is to find the position start for each pattern, sort based on position and then extract to the next start.  
We use each pattern's start position to guide extraction so there is no _trimming_ of the text


In [9]:
# patterns is all caps as a Python convention
# we define the object once and do not touch it inside a function
# It acts like a constant 
PATTERNS = [
    ('Is central bank prohibited from buying/selling government securities in primary market', 'primmkt'),
    ('Interest rates on loans must be?', 'interest'),
    ('Maturity of loans', 'maturity'),
    ('Limits on central bank lending determined by', 'limits_on_lending'),
    ('Potential borrowers from bank', 'potential_borrowers'),
    ('Terms of lending', 'termsoflend'),
    ('Securitized lending', 'securitized'),
    ('Advances ', 'advances'),
    ('Objectives ', 'objectives'),
    ('Role in government', 'budget'),
    ('Resolution of conflict', 'resolution'),
    ('Who formulates monetary policy?', 'whoformulates'),
    ('May CEO hold other offices in government?', 'otheroffices'),
    ('Dismissal', 'dismissal'),
    ('Who appoints CEO?', 'whoappts'),
    ('Term of office', 'termofoffice')]

# Naming objects

Functions - snake case with descriptive name (verb and subject)  
Constants - all capital letters  
objects - descriptive lower case names  
Iterators — short names or single letters are acceptable for generic counters (for i in range(10)), but use descriptive names when the variable has meaning (for country_name, country_text in slices)  
classes - CamelCase 

In [10]:
def extract_cbi_section(slices):
    # Dictionary to store extracted sections for all countries
    all_extracted_sections = {}
    # Loop over each country slice in slices
    for country_name, country_text in slices:
        # Dictionary to store the extracted sections for this country
        extracted_sections = {}
        # Loop through patterns and extract sections for the current country's text
        for pattern, label in PATTERNS:
            index = country_text.find(pattern)
            if index != -1:
                # Store the slice in the extracted_sections for the current country
                extracted_sections[label] = country_text[index:]
                country_text = country_text[:index]  # Update text to exclude found pattern and everything after it
        # Add the current country's extracted sections to the all_extracted_sections dictionary
        all_extracted_sections[country_name] = extracted_sections
    return all_extracted_sections

In [11]:
scores = extract_cbi_section(cty_slices)

In [12]:
print(len(scores))
scores['SAMOA']

32


{'primmkt': 'Is central bank prohibited from buying/selling government securities in primary market? (.025)\nSamoa 2008: The Bank may acquire notes, bills, securities or other evidence of debt issued or guarantied by the Government, offered for sale to the public or part of a public issued. (Section 44) (coded as 0)\n \n',
 'interest': 'Interest rates on loans must be? (.025) \nSamoa 2008: No mention of interest rate – coded as .25\n (h) ',
 'maturity': 'Maturity of loans (.025) \nSamoa 2008: The Bank may make advances to the Government: \n(a) By overdraft facility in anticipation of current budget revenue, repayable not later than the end of the financial year within which an advance is made and the total of moneys advanced shall not exceed at any time 25% of the total revenue received into the Treasury Fund in the preceding financial year; and \n(b) In respect of any payment relating to the membership of Samoa in any international financial institutions, on terms and conditions propo

# Final steps

The last part of the process is then to extract:
- country-year
- text
- section or article
- coding value

I use a similar logic as before to extract each part.  
I extract one part and return the rest as the remaining string.  
Once this is done for coding value, article and country-year, the remaining part will be the text. 


In [13]:
    
PATTERN_DICT = {pv: re.compile('^' + pk + '.*',re.I) for pk, pv in PATTERNS}
CODED_REGEX = re.compile(r'[( ]Coded as.+',re.I)
ARTICLES_REGEX = re.compile(r'[( ](Article |Section |Art\. ).{1,7}$',re.I)
CTYYR_REGEX = re.compile(r'^[^:]+',re.I)
#for s in sections:
#    s, coded = regex_split(s, CODED_REGEX, 'forward')
#    s, articles = regex_split(s, ARTICLES_REGEX, 'forward')
#    s, ctyyr = regex_split(s, CTYYR_REGEX, 'back')


In [14]:
def regex_split(section,re_pattern,how):
    c = re.search(re_pattern,section)
    matchgp=''
    if c:
        if how != 'back':
            section = section[:c.start()].strip()
        elif how=='back':
            section = section[c.end()+1:].strip()
        matchgp = c.group()
        matchgp = re.sub('(^\(|\)$)','',matchgp)
    return section.strip(), matchgp     
